# Universe — step 2 of 8

**In plain words:** the eligible list, rebuilt for each date rather than for today.

**It produces** `Universe/Security_Master.csv` and `Universe/Data_Issues.csv`.

**It prevents** survivorship bias — testing on the winners that happened to survive.

> **Runs after `Data/curator.py`**, because it profiles the files that were downloaded, and
> **before `Data/refinery.py`**, because that stage joins the security master this one writes.

**The universe is decided in `Universe/` and nowhere else** — the seed says which securities, this
notebook what each one is and from when. Nothing downstream second-guesses either.

```
Universe/Investable_Universe.csv   the seed, committed  ->  edit this to change the universe
        |
        +--> Data/curator.py       downloads one file per identifier in it
        |
        +--> this notebook         Security_Master.csv, Data_Issues.csv
```

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## 0 · Setup

Paths and the provider key, read from `Config/.env`. Nothing here touches the network, and no
value from that file is ever printed.

## The seed

`Universe/Investable_Universe.csv` is the only thing that decides what this repository is about.
Replace its rows with equities, ETFs, FX crosses, crypto pairs or futures and every stage below
still runs — nothing downstream names an asset class.

**One column is required: `main_identifier`**, the name the Data Curator asks a provider for.
Every other column is yours. Most strategies want at least a readable name and whatever they group
securities by; add those columns here and they flow through the whole pipeline.

In [ ]:
# EXAMPLE-ONLY CELL
import csv
import datetime
import json
import os
import pathlib
import urllib.parse
import urllib.request

import pandas

import kaxanuk.data_curator

# Every path below is relative to the repository root, whichever folder the notebook was started in.
NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = (
    NOTEBOOK_DIRECTORY
    if (NOTEBOOK_DIRECTORY / "Universe").is_dir()
    else NOTEBOOK_DIRECTORY.parent
)
os.chdir(REPOSITORY_ROOT)

SEED_PATH = pathlib.Path("Universe/Investable_Universe.csv")
CURATOR_DIRECTORY = pathlib.Path("Data/Curator/Time_Series")
PROVIDER_CACHE_PATH = pathlib.Path("Universe/Provider_Cache/profiles.json")
SECURITY_MASTER_PATH = pathlib.Path("Universe/Security_Master.csv")
DATA_ISSUES_PATH = pathlib.Path("Universe/Data_Issues.csv")
BENCHMARK_HOLDINGS_PATH = pathlib.Path("Data/Curator/Benchmarks/KN_US_Equity_Benchmark.csv")

# The rule holds thirty names and needs 200 trading days of prices before it can rank anything.
BOOK_SIZE = 30
WARM_UP_DAYS = 200

# The key is loaded into the environment and read by name only; no value is printed anywhere here.
kaxanuk.data_curator.load_config_env()

seed = pandas.read_csv(SEED_PATH)
print(f"seed: {len(seed)} identifiers from {SEED_PATH}")
seed.head()

## 1 · The seed, checked

Two things to establish before anything downstream trusts this file.

1. **Every identifier is unique**, and so is every other identity column you carry. A repeated
   identity under two rows is usually a renamed security, and the two legs have to be stitched
   into one position or the book holds it twice. A point-in-time universe is supposed to contain
   these; what it must not do is hide them.
2. **The grouping is complete.** Whatever column the strategy compares things by, a missing value
   in it is a security that silently drops out of every group-level view.

In [ ]:
# EXAMPLE-ONLY CELL
# 1. Every identifier unique. A repeat would be two rows for one security, and the panel would
#    hold the position twice.
duplicates = seed["main_identifier"][seed["main_identifier"].duplicated()]
print(f"duplicate identifiers: {len(duplicates)}")

# 2. The grouping is complete. This seed carries identity alone, so there is nothing to group by
#    yet: the strategy groups on `current_sector`, which the master below supplies and the refinery
#    joins. A seed that carried its own grouping column would be checked for blanks here.
print(f"columns in the seed: {list(seed.columns)}")
print(f"blank identifiers: {int(seed['main_identifier'].isna().sum())}")

## 2 · The security master

The seed gives identity. Everything else comes from the provider: the official name, what kind of
instrument it is, where it trades, in what currency, and when it started.

**Cache the provider's raw payload, then shape the master from the cache.** Re-running then costs
nothing, changing the column mapping never triggers a refetch, and the untouched payload stays
available for fields this notebook does not yet use. Adding a provider is one fetch function and
one normaliser.

**The seed wins.** Its identity columns define the universe, so a provider value never overwrites
one — it is compared instead, and a disagreement is reported. A provider that now points a symbol
at a different security is a recycled identifier, and joining on it across the whole history would
silently mix two companies.

> **What the master carries is what a security is *today*.** The provider keeps no history, so on
> a universe whose members get reclassified, every period before the move is attributed wrongly and
> nothing raises an error. That is why the refinery prefixes every joined column `current_`.

In [ ]:
# EXAMPLE-ONLY CELL
# The provider is asked once per identifier and the raw payload is cached, so re-running this
# notebook costs nothing and changing the mapping below never triggers a refetch.
PROVIDER_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
cached_profiles = (
    json.loads(PROVIDER_CACHE_PATH.read_text(encoding="utf-8"))
    if PROVIDER_CACHE_PATH.is_file()
    else {}
)
identifiers = list(seed["main_identifier"])
pending = [identifier for identifier in identifiers if identifier not in cached_profiles]
print(f"cached: {len(cached_profiles)}, to fetch: {len(pending)}")

# FMP's stable profile endpoint takes one symbol per request. The cache is written every fifty,
# so an interrupted run resumes where it stopped.
for position, identifier in enumerate(pending, start=1):
    request_url = (
        "https://financialmodelingprep.com/stable/profile?symbol="
        + urllib.parse.quote(identifier)
        + "&apikey="
        + os.environ["KNDC_API_KEY_FMP"]
    )
    with urllib.request.urlopen(request_url, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))

    for row in payload:
        cached_profiles[row["symbol"]] = row

    cached_profiles.setdefault(identifier, {})

    if position % 50 == 0:
        PROVIDER_CACHE_PATH.write_text(json.dumps(cached_profiles, indent=1), encoding="utf-8")

PROVIDER_CACHE_PATH.write_text(json.dumps(cached_profiles, indent=1), encoding="utf-8")
print(f"payloads cached: {len(cached_profiles)}")

In [ ]:
# EXAMPLE-ONLY CELL
# The seed wins on identity: the provider's symbol is compared, never written over the seed's.
master_rows = []
identity_disagreements = []

for identifier in identifiers:
    profile = cached_profiles.get(identifier) or {}

    if profile.get("symbol") not in (None, identifier):
        identity_disagreements.append((identifier, profile.get("symbol")))

    master_rows.append({
        "main_identifier": identifier,
        "name": profile.get("companyName"),
        "instrument_type": "etf" if profile.get("isEtf") else "equity",
        "exchange": profile.get("exchange"),
        "currency": profile.get("currency"),
        "inception": profile.get("ipoDate"),
        "isin": profile.get("isin"),
        "is_actively_trading": profile.get("isActivelyTrading"),
        # Today's classification, never point-in-time: the refinery joins these as `current_*`.
        "sector": profile.get("sector"),
        "industry": profile.get("industry"),
    })

security_master = pandas.DataFrame(master_rows)
security_master.to_csv(SECURITY_MASTER_PATH, index=False)
print(f"wrote {SECURITY_MASTER_PATH}: {len(security_master)} rows")
print(f"identifiers the provider does not carry: {int(security_master['name'].isna().sum())}")
print(f"identity disagreements (recycled symbols): {len(identity_disagreements)}")
security_master.head()

## 3 · Composition, and how long each member has existed

Two facts decide what a backtest can honestly claim. **What the universe is made of** sets what
diversification is even available; **when each member started trading** sets the window, because a
universe is only complete from the inception of its youngest member.

## 4 · The data-issues register

The seed says what *should* exist. This section reads what *does*, and writes
`Universe/Data_Issues.csv`. Eight checks, each of which has cost somebody real time somewhere:

| Check | The failure it catches |
| --- | --- |
| **Missing file** | an identifier the provider does not carry, which becomes a silent hole in the panel |
| **Schema drift** | a folder holding two column sets, which makes every downstream read conditional |
| **No usable signal** | a history shorter than the strategy's longest warm-up, so the name can never be selected |
| **Unusable values** | zero or negative prices, which break every return calculation downstream |
| **Impossible daily move** | an adjusted price that multiplies by more than six in a day: an unadjusted corporate action or a bad print, not a return |
| **Late start** | a series that begins after the panel does, so the cross-section is smaller before that date |
| **Status disagreement** | a series that ends early on a name the provider still calls active: a data gap, not a delisting |
| **Early end** | a series that stops early — delisted or halted, and a held position must be exited on its last priced day |

Two checks the `universe-point-in-time` skill lists are not written here: **internal gaps**, and
**identity conflict**, because the provider cache is keyed by the symbol the provider returns, so
the comparison in section 2 cannot disagree, and this seed carries no second identity column to
check it against.

In [ ]:
# EXAMPLE-ONLY CELL
# What the universe is made of, and how much of it is already dead. A seed showing 0% delisted
# would not be a universe, it would be a survivor list.
delisted = security_master["is_actively_trading"] == False  # noqa: E712 - None must not count
print(f"delisted or no longer trading: {int(delisted.sum())} of {len(security_master)}")
print(security_master["sector"].value_counts(dropna=False).head(15))
print(security_master["instrument_type"].value_counts(dropna=False))

inception = pandas.to_datetime(security_master["inception"], errors="coerce")
print(f"listed before 2002: {int((inception < '2002-01-01').sum())}")
print(f"listed after 2017: {int((inception > '2017-01-01').sum())}")

In [ ]:
# EXAMPLE-ONLY CELL
# Eight checks against what the curator actually wrote. Each row names the identifiers, so the next
# stage gets a work list rather than a chart to interpret.
# The window the long run opens on, not the window the download asked for: every series starts
# after 2001-01-01, so measuring against that would flag all 788 and tell nobody anything.
LONG_WINDOW_START = pandas.Timestamp("2002-01-01")
PANEL_END = pandas.Timestamp("2026-06-01")

price_dates = {}
short_history = []
non_positive_prices = []
impossible_moves = []
IMPOSSIBLE_DAILY_MOVE = 5.0
expected_header = None
schema_drift = []

for identifier in identifiers:
    path = CURATOR_DIRECTORY / f"{identifier}.csv"

    if not path.is_file():
        continue

    frame = pandas.read_csv(
        path,
        usecols=["m_date", "m_close_dividend_and_split_adjusted"],
        parse_dates=["m_date"],
    )
    price_dates[identifier] = frame

    with path.open(encoding="utf-8-sig", newline="") as handle:
        header = tuple(next(csv.reader(handle), []))

    if expected_header is None:
        expected_header = header
    elif header != expected_header:
        schema_drift.append(identifier)

    if len(frame) < WARM_UP_DAYS:
        short_history.append(identifier)

    if (frame["m_close_dividend_and_split_adjusted"] <= 0).any():
        non_positive_prices.append(identifier)

    # A price that multiplies by six in one day is an unadjusted corporate action or a bad print,
    # not a return. Real squeezes reach three or four times; the threshold sits above them on
    # purpose, so a flag here means the series is wrong rather than merely wild.
    daily_move = frame["m_close_dividend_and_split_adjusted"].pct_change(fill_method=None)

    if (daily_move.abs() > IMPOSSIBLE_DAILY_MOVE).any():
        impossible_moves.append(identifier)

missing_files = [
    identifier
    for identifier in identifiers
    if not (CURATOR_DIRECTORY / f"{identifier}.csv").is_file()
]
late_start = [
    identifier
    for identifier, frame in price_dates.items()
    if frame["m_date"].min() > LONG_WINDOW_START
]
early_end = [
    identifier
    for identifier, frame in price_dates.items()
    if frame["m_date"].max() < PANEL_END - datetime.timedelta(days=7)
]
# A series that stops while the provider still calls the security active is a data gap, not a
# delisting -- and the two need different answers, so they are counted apart.
active = set(security_master.loc[security_master["is_actively_trading"] == True, "main_identifier"])  # noqa: E712
status_disagreement = [
    identifier
    for identifier in early_end
    if identifier in active
]

issues = pandas.DataFrame([
    {
        "check": "missing file",
        "severity": "blocking",
        "count": len(missing_files),
        "identifiers": " ".join(missing_files),
    },
    {
        "check": "schema drift",
        "severity": "blocking",
        "count": len(schema_drift),
        "identifiers": " ".join(schema_drift),
    },
    {
        "check": "no usable signal (history shorter than the warm-up)",
        "severity": "blocking",
        "count": len(short_history),
        "identifiers": " ".join(short_history),
    },
    {
        "check": "unusable values (zero or negative prices)",
        "severity": "blocking",
        "count": len(non_positive_prices),
        "identifiers": " ".join(non_positive_prices),
    },
    {
        "check": "impossible daily move (the adjusted price multiplies by more than six)",
        "severity": "blocking",
        "count": len(impossible_moves),
        "identifiers": " ".join(impossible_moves),
    },
    {
        "check": "late start (no prices when the long window opens in 2002)",
        "severity": "expected",
        "count": len(late_start),
        "identifiers": " ".join(late_start[:50]),
    },
    {
        "check": "status disagreement (series ends early, provider still calls it active)",
        "severity": "blocking",
        "count": len(status_disagreement),
        "identifiers": " ".join(status_disagreement),
    },
    {
        "check": "early end (last price before the panel ends: delisted, acquired or halted)",
        "severity": "expected",
        "count": len(early_end),
        "identifiers": " ".join(early_end[:50]),
    },
])
issues = issues.sort_values("severity")
issues.to_csv(DATA_ISSUES_PATH, index=False)
print(f"wrote {DATA_ISSUES_PATH}")
issues[["check", "severity", "count"]]

## 5 · When the universe is actually usable

A file that starts in 2010 gives no signal in 2010. Every feature has a warm-up, and a five-year
one moves the honest start of a backtest by five years.

**The date that matters is the first day on which every security can be both priced and
signalled.** Before it the strategy is choosing from a smaller menu than it appears to be, and a
backtest that starts earlier is quietly comparing books drawn from different universes. Nothing
else in the pipeline says so, which is why it is answered here.

In [ ]:
# EXAMPLE-ONLY CELL
# A security can be signalled from its 200th trading day. The universe is usable from the first
# date on which enough of them can be signalled at once to fill the book.
first_signal_dates = {
    identifier: frame["m_date"].iloc[WARM_UP_DAYS - 1]
    for identifier, frame in price_dates.items()
    if len(frame) >= WARM_UP_DAYS
}
last_price_dates = {
    identifier: frame["m_date"].max()
    for identifier, frame in price_dates.items()
}

calendar = pandas.DatetimeIndex(
    sorted({date for frame in price_dates.values() for date in frame["m_date"]})
)
signallable = pandas.Series(0, index=calendar, dtype=int)

for identifier, first_date in first_signal_dates.items():
    window = (calendar >= first_date) & (calendar <= last_price_dates[identifier])
    signallable[window] += 1

deep_enough = signallable[signallable >= BOOK_SIZE]
print(f"securities that can ever be signalled: {len(first_signal_dates)} of {len(identifiers)}")
print(f"first date with {BOOK_SIZE} signallable securities: {deep_enough.index.min().date()}")

# The index's own holdings start later than its prices do, so membership decides the honest start
# of the point-in-time run.
holdings = pandas.read_csv(BENCHMARK_HOLDINGS_PATH, parse_dates=["date_column"], dayfirst=True)
members_per_date = (holdings.drop(columns="date_column") > 0).sum(axis=1)
first_holding_date = holdings["date_column"].min().date()
last_holding_date = holdings["date_column"].max().date()
print(f"index holdings run {first_holding_date} to {last_holding_date}")
print(f"members per date: {int(members_per_date.min())} to {int(members_per_date.max())}")
print(f"usable with dated membership: {first_holding_date}")

## 6 · Handoff

| Output | Consumed by |
| --- | --- |
| `Universe/Security_Master.csv` | `Data/refinery.py`, which joins its columns onto the panel |
| `Universe/Data_Issues.csv` | the caveats section of every `FINDINGS_N.md` |

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Classification is a snapshot, not a history.** | A security reclassified mid-window is misattributed before its move. The `current_*` prefix marks exactly this. |
| 2 | **The last day of a delisted name is unaudited.** Nothing here checks whether a truncated series ends on a real final price or on a provider gap. | On a universe that retains delisted names, the missing returns are disproportionately the bad ones. |
| 3 | **Inception dates come from the provider, not from the price file.** | The price file is what the backtest actually trades; the master is what a reader believes. Where they disagree, say so. |

In [ ]:
# EXAMPLE-ONLY CELL
print(f"{SECURITY_MASTER_PATH}: {len(security_master)} rows -> Data/refinery.py joins as current_*")
print(f"{DATA_ISSUES_PATH}: {len(issues)} checks -> the caveats table of FINDINGS_1.md")
print(f"{PROVIDER_CACHE_PATH}: {len(cached_profiles)} cached payloads, regenerable and gitignored")

## 7 · Verify

**Assertions that raise when this stage's output is wrong**, read back from the files this notebook
wrote rather than from the variables that wrote them. A check that prints is one somebody has to
read; one that raises stops the run, so a notebook that reaches its last cell is one whose outputs
hold. At the least:

- `Universe/Security_Master.csv` has one row per identifier in the seed, each once, and no
  identifier the seed lacks;
- `Universe/Data_Issues.csv` has a row for every check section 4 runs, each with a count;
- the downloaded files and the register's missing ones account for the whole seed;
- section 5 found a date from which the universe is usable.

In [ ]:
# EXAMPLE-ONLY CELL
# Read back what this notebook wrote, and raise on the first thing that is wrong. A printed PASS is
# read by whoever is watching; a raised error stops a run nobody is.
written_master = pandas.read_csv(SECURITY_MASTER_PATH)
written_issues = pandas.read_csv(DATA_ISSUES_PATH).set_index("check")
missing_count = int(written_issues.loc["missing file", "count"])
repeated_identifiers = written_master["main_identifier"].duplicated()
verifications = {
    "the master has one row per seed identifier": len(written_master) == len(seed),
    "no identifier appears twice in the master": not bool(repeated_identifiers.any()),
    "the master holds exactly the seed's identifiers": (
        set(written_master["main_identifier"]) == set(seed["main_identifier"])
    ),
    "the register holds all eight checks": len(written_issues) == 8,
    "every check has a count": written_issues["count"].notna().all(),
    "files and missing files account for the whole seed": (
        len(price_dates) + missing_count == len(identifiers)
    ),
    "a date exists on which the book can be filled": len(deep_enough) > 0,
}
failed = [
    name
    for name, passed in verifications.items()
    if not passed
]

if len(failed) > 0:
    message = f"the universe failed verification: {'; '.join(failed)}"

    raise AssertionError(message)

print(f"verified: {len(verifications)} checks on {SECURITY_MASTER_PATH} and {DATA_ISSUES_PATH}")